In [1]:
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
import pandas as pd
import time

print("modules imported succesfully")

/Users/chaitanyayadav/personal/code/ai_projects/hf_models/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


modules imported succesfully


In [2]:
class StudentProfile(BaseModel):
    degree_level: str = Field(description="student's current degree level")
    percentage: float = Field(description="student's latest percentage in the degree")
    is_reserved: bool = Field(desctiption="whether student belong to a reserved category or not")
    academic_background: str = Field(description="student's degree name and subject")
    subjects: List[str] = Field(description="subjects studied in the course")
    interests: List[str] = Field(description="student's area of interests")
    career_goal: str = Field(desctiption="student's career goal")
    preferred_skills: List[str] = Field(description="student's preferred skills to learn")
    preferred_domain: str = Field(description="student's preferred domain")

print("class initialised successfully")

class initialised successfully


/var/folders/0n/3157pzwj13xd5kvdtpbqgv440000gn/T/ipykernel_67522/3381895869.py:4: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desctiption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  is_reserved: bool = Field(desctiption="whether student belong to a reserved category or not")
/var/folders/0n/3157pzwj13xd5kvdtpbqgv440000gn/T/ipykernel_67522/3381895869.py:8: PydanticDeprecatedSince20: Using extra keyword arguments on `Field` is deprecated and will be removed. Use `json_schema_extra` instead. (Extra keys: 'desctiption'). Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  career_goal: str = Field(desctiption="student's career goal")


In [4]:
system_prompt = """
You are a student profile generator for an Indian university recommendation system.

Your task is to generate REALISTIC and DIVERSE student profiles that represent actual students seeking postgraduate admissions in India.

================================
OUTPUT REQUIREMENTS
================================
Return ONLY valid JSON matching the exact StudentProfile schema provided.
No explanations, no commentary, no markdown formatting.

================================
FIELD-BY-FIELD GENERATION RULES
================================

1. degree_level (str)
-------------------
Current educational qualification of the student.
Must be EXACTLY one of: "PreUG", "UG", "PG"

Mapping:
- "PreUG" → Student has completed 10+2 / Higher Secondary (looking for UG programs)
- "UG" → Student has completed Bachelor's degree (looking for PG programs)
- "PG" → Student has completed Master's degree (looking for PhD programs)

Distribution to maintain diversity:
- 60% should be "UG" (most common - looking for Master's programs)
- 35% should be "PreUG" (looking for Bachelor's programs)
- 5% should be "PG" (looking for PhD programs)

2. percentage (float)
------------------
Student's marks in their latest completed degree.
Return as float between 40.0 and 95.0

Realistic distribution:
- 40-49%: 5% of students (struggling students)
- 50-59%: 25% of students (average students)
- 60-74%: 45% of students (good students)
- 75-84%: 20% of students (very good students)
- 85-95%: 5% of students (exceptional students)

Examples: 58.5, 72.0, 81.3, 45.8, 89.2

3. is_reserved (bool)
------------------
Whether student belongs to reserved category (SC/ST/OBC/PwD).
Use true or false (JSON boolean, lowercase)

Distribution:
- 30% should be true (reserved category)
- 70% should be false (general category)

4. academic_background (str)
-------------------------
Current degree name and major subject(s).
Format: "[Degree Type] in [Subject/Specialization]"

Examples for different degree levels:

PreUG students:
- "12th Science (PCM)" (Physics, Chemistry, Maths)
- "12th Science (PCB)" (Physics, Chemistry, Biology)
- "12th Commerce"
- "12th Arts (Humanities)"

UG students (most common):
- "Bachelor of Science in Physics"
- "Bachelor of Technology in Computer Science"
- "Bachelor of Arts in English Literature"
- "Bachelor of Commerce"
- "Bachelor of Engineering in Mechanical Engineering"
- "B.Sc. in Biotechnology"
- "B.A. in Economics"
- "BBA (Business Administration)"

PG students (rare):
- "Master of Science in Chemistry"
- "Master of Technology in Data Science"
- "M.A. in History"
- "MBA (Marketing)"

Keep it concise and realistic.

5. subjects (List[str])
--------------------
3-6 major subjects studied in their current degree.
Use actual subject names, not descriptions.

Examples by background:

Science students:
["Physics", "Chemistry", "Mathematics"]
["Biology", "Chemistry", "Zoology"]
["Biotechnology", "Microbiology", "Genetics"]

Engineering students:
["Data Structures", "Algorithms", "Database Systems"]
["Thermodynamics", "Fluid Mechanics", "Machine Design"]

Commerce students:
["Accounting", "Economics", "Business Studies"]
["Finance", "Marketing", "Statistics"]

Arts/Humanities students:
["English Literature", "History", "Political Science"]
["Psychology", "Sociology", "Philosophy"]

6. interests (List[str])
---------------------
3-5 areas the student is genuinely interested in pursuing.
These should be SPECIFIC, not vague.

Good examples:
["Machine Learning", "Data Analysis", "Artificial Intelligence"]
["Genetic Engineering", "Molecular Biology", "Drug Development"]
["Financial Markets", "Investment Banking", "Corporate Finance"]
["Environmental Conservation", "Climate Change", "Sustainable Development"]
["Digital Marketing", "Content Strategy", "Brand Management"]

Bad examples (too vague):
["Technology", "Science", "Business"]  # Too broad!

7. career_goal (str)
-----------------
Student's primary career aspiration. Be SPECIFIC and realistic.
Format: Single sentence describing the role/field they want.

Good examples:
"Become a data scientist in a tech company"
"Work as a biotechnology researcher in pharmaceutical industry"
"Join civil services and work in policy making"
"Become a financial analyst in investment banking"
"Work as an environmental consultant"
"Pursue research in artificial intelligence"
"Start a tech startup in fintech domain"
"Become a clinical psychologist"

Bad examples:
"Get a good job"  # Too vague!
"Be successful"  # Not specific!

8. preferred_skills (List[str])
----------------------------
3-5 specific skills the student wants to learn/develop.
These should align with their career goals.

Examples by career goal:

Tech/Data Science career:
["Python Programming", "Machine Learning", "Deep Learning", "Data Visualization"]

Research career:
["Research Methodology", "Statistical Analysis", "Laboratory Techniques", "Scientific Writing"]

Business career:
["Financial Modeling", "Market Research", "Business Strategy", "Leadership Skills"]

Healthcare career:
["Clinical Skills", "Patient Care", "Medical Research", "Diagnostics"]

9. preferred_domain (str)
----------------------
Broad academic domain the student wants to pursue.
Must be EXACTLY one of these standardized values:

"Life Sciences"
"Science"
"Engineering"
"Management"
"Commerce"
"Humanities"
"Law"
"Education"
"Arts"
"Health Sciences"
"Agriculture"
"Social Sciences"

Choose based on career goal and background.

================================
DIVERSITY REQUIREMENTS
================================

Generate students with VARIED profiles:
- Different degree levels (mostly UG, some PreUG, few PG)
- Different academic backgrounds (Science, Commerce, Arts, Engineering)
- Different marks percentages (realistic distribution)
- Mix of reserved and general category
- Different career aspirations
- Domain transitions (e.g., Engineering student wanting Management, Science student wanting Data Science)

================================
REALISM REQUIREMENTS
================================

1. **Consistency**: Ensure all fields are internally consistent
   - If academic_background is "B.Tech in Computer Science", subjects should be tech-related
   - If career_goal is "data scientist", preferred_skills should include programming/ML
   - If degree_level is "PreUG", academic_background should be "12th..."

2. **Indian Context**: Use Indian education system terminology
   - Use "12th", not "Grade 12" or "Senior Secondary"
   - Use "Bachelor of Science" or "B.Sc.", not "BSc" or "BS"
   - Use realistic Indian college subjects and streams

3. **Career Goals**: Should match typical aspirations of Indian students
   - Government jobs (UPSC, banking, PSUs)
   - Tech industry (software developer, data scientist)
   - Research (scientist, professor)
   - Traditional professions (doctor, engineer, lawyer)
   - Entrepreneurship (startup founder)

================================
EXAMPLES OF GOOD PROFILES
================================

Example 1 (Typical UG student seeking PG):
{
  "degree_level": "UG",
  "percentage": 68.5,
  "is_reserved": false,
  "academic_background": "Bachelor of Science in Biotechnology",
  "subjects": ["Genetics", "Microbiology", "Biochemistry", "Molecular Biology"],
  "interests": ["Genetic Engineering", "Drug Development", "Medical Research"],
  "career_goal": "Work as a biotechnology researcher in pharmaceutical industry",
  "preferred_skills": ["Genetic Analysis", "Laboratory Techniques", "Research Methodology"],
  "preferred_domain": "Life Sciences"
}

Example 2 (PreUG student seeking UG):
{
  "degree_level": "PreUG",
  "percentage": 82.0,
  "is_reserved": true,
  "academic_background": "12th Science (PCM)",
  "subjects": ["Physics", "Chemistry", "Mathematics"],
  "interests": ["Data Science", "Artificial Intelligence", "Programming"],
  "career_goal": "Become a data scientist in tech industry",
  "preferred_skills": ["Python Programming", "Machine Learning", "Statistical Analysis"],
  "preferred_domain": "Engineering"
}

Example 3 (Career transition - Commerce to Management):
{
  "degree_level": "UG",
  "percentage": 71.2,
  "is_reserved": false,
  "academic_background": "Bachelor of Commerce",
  "subjects": ["Accounting", "Economics", "Finance", "Business Studies"],
  "interests": ["Financial Markets", "Investment Banking", "Business Strategy"],
  "career_goal": "Join investment banking as a financial analyst",
  "preferred_skills": ["Financial Modeling", "Market Research", "Data Analysis"],
  "preferred_domain": "Management"
}

Example 4 (Arts student seeking Social Sciences):
{
  "degree_level": "UG",
  "percentage": 54.3,
  "is_reserved": true,
  "academic_background": "Bachelor of Arts in Psychology",
  "subjects": ["Psychology", "Sociology", "Statistics"],
  "interests": ["Mental Health", "Counseling", "Community Development"],
  "career_goal": "Work as a counselor in educational institutions",
  "preferred_skills": ["Counseling Techniques", "Psychological Assessment", "Research Methods"],
  "preferred_domain": "Education"
}

================================
CRITICAL REMINDERS
================================

1. Return ONLY valid JSON - no text before or after
2. Use exact field names from the schema
3. Maintain internal consistency across all fields
4. Generate diverse profiles - avoid patterns
5. Use realistic Indian educational context
6. Ensure career goals match interests and preferred skills
7. Use proper JSON formatting (lowercase true/false for booleans)

Now generate a realistic student profile following all these rules.
"""

In [ ]:
gen_model = ChatOllama(model="qwen2.5:7b", temperature=0.7).with_structured_output(StudentProfile)

results = []

for i in range(20):

    conversation = [
        SystemMessage(content=system_prompt),
    ]

    try:
        response: StudentProfile = gen_model.invoke(conversation)
        results.append(response.model_dump())
        print(f"Student profile generated : {i+1}")
        time.sleep(0.5)
    except Exception as e:
        print(f"Some error occured while generation: {str(e)}")

df = pd.DataFrame(results)
df.to_csv("students_data.csv", index=False)
    
    

Student profile generated : 1
Student profile generated : 2
Student profile generated : 3
Student profile generated : 4
Student profile generated : 5
Student profile generated : 6
Student profile generated : 7
Student profile generated : 8
Student profile generated : 9
Student profile generated : 10
Student profile generated : 11
Student profile generated : 12
Student profile generated : 13
Student profile generated : 14
Student profile generated : 15
Student profile generated : 16
Student profile generated : 17
Student profile generated : 18
Student profile generated : 19
Student profile generated : 20
